# Multiple Linear Regression

**The big picture:** the data has 50 startups. For each one we know how much it spent on **R&D**, **Administration** and **Marketing**, which **State** it is in, and its **Profit**. The goal is to predict profit from all of those inputs at once.

With one input (simple linear regression) the model finds a straight line. With several inputs the rule becomes:

> **Profit = (base amount) + (a × R&D) + (b × Admin) + (c × Marketing) + (an adjustment for the State)**

The program's job is to work out the base amount and each multiplier from the data.

### Before you start
- **Read first:** **Simple Linear Regression** (the same model with one input) and **Data Preprocessing Tools** (for one-hot encoding, used here on State).
- **New words in this notebook:** *multiple linear regression*, *coefficients*, *dummy variables*, *dummy variable trap*, *backward elimination*. Look them up in the [glossary](../../../GLOSSARY.md) if you get stuck.

## Importing the libraries

- **pandas** reads and handles tables, like Excel in Python.
- **numpy** does fast maths on lists of numbers.
- **matplotlib** draws charts. It is imported out of habit but not used here: with four inputs there is no simple 2D chart to draw.

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Importing the dataset

- Reads the CSV file into a table.
- **X** = every column except the last (R&D Spend, Administration, Marketing Spend, State). These are the **inputs**.
- **y** = the last column, *Profit*. This is what we want to **predict**.

`print(X)` just shows what X looks like. Note that the State column still holds words like "New York".

In [0]:
dataset = pd.read_csv('50_Startups.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

In [3]:
print(X)

[[165349.2 136897.8 471784.1 'New York']
 [162597.7 151377.59 443898.53 'California']
 [153441.51 101145.55 407934.54 'Florida']
 [144372.41 118671.85 383199.62 'New York']
 [142107.34 91391.77 366168.42 'Florida']
 [131876.9 99814.71 362861.36 'New York']
 [134615.46 147198.87 127716.82 'California']
 [130298.13 145530.06 323876.68 'Florida']
 [120542.52 148718.95 311613.29 'New York']
 [123334.88 108679.17 304981.62 'California']
 [101913.08 110594.11 229160.95 'Florida']
 [100671.96 91790.61 249744.55 'California']
 [93863.75 127320.38 249839.44 'Florida']
 [91992.39 135495.07 252664.93 'California']
 [119943.24 156547.42 256512.92 'Florida']
 [114523.61 122616.84 261776.23 'New York']
 [78013.11 121597.55 264346.06 'California']
 [94657.16 145077.58 282574.31 'New York']
 [91749.16 114175.79 294919.57 'Florida']
 [86419.7 153514.11 0.0 'New York']
 [76253.86 113867.3 298664.47 'California']
 [78389.47 153773.43 299737.29 'New York']
 [73994.56 122782.75 303319.26 'Florida']
 [67532

## Encoding categorical data

The model can only do maths on numbers, and "California" is not a number. Replacing the states with 1, 2 and 3 would wrongly tell the model that Florida is "3 times" New York.

**One-hot encoding** replaces the State column with **three yes/no columns**:

| State | California | Florida | New York |
|---|---|---|---|
| New York | 0 | 0 | 1 |
| California | 1 | 0 | 0 |
| Florida | 0 | 1 | 0 |

- `[3]` means "apply this to column 3" (counting from 0), which is State.
- `remainder='passthrough'` means "leave the other columns as they are".
- The new state columns are placed **at the front** of X, which you can see in the second `print(X)`.

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [3])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

In [5]:
print(X)

[[0.0 0.0 1.0 165349.2 136897.8 471784.1]
 [1.0 0.0 0.0 162597.7 151377.59 443898.53]
 [0.0 1.0 0.0 153441.51 101145.55 407934.54]
 [0.0 0.0 1.0 144372.41 118671.85 383199.62]
 [0.0 1.0 0.0 142107.34 91391.77 366168.42]
 [0.0 0.0 1.0 131876.9 99814.71 362861.36]
 [1.0 0.0 0.0 134615.46 147198.87 127716.82]
 [0.0 1.0 0.0 130298.13 145530.06 323876.68]
 [0.0 0.0 1.0 120542.52 148718.95 311613.29]
 [1.0 0.0 0.0 123334.88 108679.17 304981.62]
 [0.0 1.0 0.0 101913.08 110594.11 229160.95]
 [1.0 0.0 0.0 100671.96 91790.61 249744.55]
 [0.0 1.0 0.0 93863.75 127320.38 249839.44]
 [1.0 0.0 0.0 91992.39 135495.07 252664.93]
 [0.0 1.0 0.0 119943.24 156547.42 256512.92]
 [0.0 0.0 1.0 114523.61 122616.84 261776.23]
 [1.0 0.0 0.0 78013.11 121597.55 264346.06]
 [0.0 0.0 1.0 94657.16 145077.58 282574.31]
 [0.0 1.0 0.0 91749.16 114175.79 294919.57]
 [0.0 0.0 1.0 86419.7 153514.11 0.0]
 [1.0 0.0 0.0 76253.86 113867.3 298664.47]
 [0.0 0.0 1.0 78389.47 153773.43 299737.29]
 [0.0 1.0 0.0 73994.56 122782.75 3

## Splitting the dataset into the Training set and Test set

- **80% of the rows (40 startups)** go into the *training set*. The model learns from these.
- **20% of the rows (10 startups)** go into the *test set*. These are held back to check the model afterwards.
- `random_state = 0` makes the shuffle come out the same every time you run it.

**Why?** Testing on data the model has never seen shows whether it actually learned the pattern rather than memorising the answers.

In [0]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

## Training the Multiple Linear Regression model on the Training set

- This is the **same class** used for simple linear regression. It sees that X has several columns and finds one multiplier per column.
- `.fit()` is the learning step. It picks the multipliers that make the gaps between predicted and real profits as small as possible (least squares).

Two related ideas you may hear about. Neither needs any action in this code:
- **The dummy variable trap:** the three state columns always add up to 1, so one is redundant. Normally you drop one, but scikit-learn's `LinearRegression` handles this for you.
- **Feature selection (e.g. backward elimination):** removing inputs that don't help. This code doesn't do it and uses all the inputs.

In [7]:
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()
regressor.fit(X_train, y_train)

LinearRegression(copy_X=True, fit_intercept=True, n_jobs=None, normalize=False)

## Predicting the Test set results

- `predict` guesses the profit for the 10 test startups the model has never seen.
- `set_printoptions(precision=2)` rounds the printout to 2 decimal places.
- `reshape(len(...), 1)` turns each flat list into a single vertical column, and `concatenate(..., 1)` glues the two columns together left to right.
- The result is a two-column table: **predicted profit | real profit**. If the numbers in each row are close, the model is doing well.

This replaces the charts from simple linear regression. With four inputs the "line" is really a flat surface in several dimensions and can't be drawn, so a side-by-side comparison is used instead.

**In one sentence:** load startup spending data, turn the State names into yes/no columns, keep 20% aside, fit one multiplier per input on the rest, then print predicted profits next to the real ones to see how close they are.

In [8]:
y_pred = regressor.predict(X_test)
np.set_printoptions(precision=2)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[103015.2  103282.38]
 [132582.28 144259.4 ]
 [132447.74 146121.95]
 [ 71976.1   77798.83]
 [178537.48 191050.39]
 [116161.24 105008.31]
 [ 67851.69  81229.06]
 [ 98791.73  97483.56]
 [113969.44 110352.25]
 [167921.07 166187.94]]


## Common mistakes to avoid
- **Encoding State as 0, 1, 2.** Using `LabelEncoder` on an input column makes the model treat State as a quantity, as if Florida were "twice" California. Use one-hot encoding for categories that have no natural order.
- **Feeding new data in the wrong column order.** After encoding, the columns are California, Florida, New York, R&D, Administration, Marketing — the state flags come *first*. A new startup passed to `predict` must use exactly that order, or the predictions will be garbage without any error.
- **Comparing raw coefficient sizes to decide what matters most.** A multiplier's size depends on the units of its column, and inputs that move together can share or swap credit. Treat coefficients as a hint, not proof of importance.
- **Assuming every input helps.** This code keeps all inputs. Some may add little, and extra weak inputs can make a model worse on new data — that's what feature selection such as backward elimination is for.

## Try this yourself
Change one thing, rerun the notebook, and see what happens.

1. After the training cell, add `print(regressor.coef_)` and `print(regressor.intercept_)`. What to notice: there are **six** multipliers, in the order California, Florida, New York, R&D, Administration, Marketing. Look at the signs and sizes of the three spending ones — each says how much predicted profit changes per extra dollar spent, with everything else held equal.
2. Right after the encoding cell, add a cell with `X = X[:, 1:]` (dropping the California column) and re-run the cells below it. What to notice: the predicted-vs-real table is essentially unchanged. California just becomes the "baseline" state, which is why scikit-learn doesn't need you to avoid the dummy variable trap by hand.
3. Restart and run from the top, but after the encoding cell add `X = X[:, [3]]` so only R&D Spend is used. What to notice: compare the new predicted-vs-real table with the original — this shows how much of the profit can be explained by R&D spending alone versus using every input.
4. Add a cell with `print(regressor.predict([[1, 0, 0, 160000, 130000, 300000]]))`. What to notice: you've predicted profit for a made-up California startup spending 160k on R&D, 130k on admin and 300k on marketing. Change the `1, 0, 0` to `0, 0, 1` (New York) and see how much the state alone shifts the answer.

## Check your understanding
Try to answer in your head first, then click to reveal.

**1. In the final printout, which column is the prediction and which is the real profit? How far off is the first row?**

<details><summary>Show answer</summary>

The left column is the predicted profit and the right is the real profit. The first row predicts 103015.2 against a real 103282.38, so it's off by only about 267.

</details>

**2. Why is there no chart like in simple linear regression?**

<details><summary>Show answer</summary>

With several inputs the model is no longer a line on a 2D chart but a flat surface in many dimensions, which can't be drawn. A predicted-vs-real table is used instead.

</details>

**3. Why doesn't the code drop one of the three state columns to avoid the dummy variable trap?**

<details><summary>Show answer</summary>

The three columns always add up to 1, so one is redundant, but scikit-learn's `LinearRegression` copes with this automatically. Dropping one by hand gives essentially the same predictions.

</details>

**4. What would happen if you skipped the encoding cell and went straight to training?**

<details><summary>Show answer</summary>

`fit` would raise an error, because the State column still contains words like 'New York' and the model can only do maths on numbers.

</details>

---
📚 [Learning path](../../../LEARNING_PATH.md) · [Glossary](../../../GLOSSARY.md)